In [26]:
# Biblioteca

import polars as pl
import numpy as np
import os

# Localização dos arquivos

pasta =  'tratados_3det'

# Retorna o nome do arquivo formatado
import re
def formatar_string(s):
   
    match = re.match(r'([a-zA-Z]+)([0-9]+(?:\.[0-9]+)?[Ee][0-9]+)', s)

    if match:
        
        palavra = match.group(1).capitalize()
        numero = match.group(2)

        return f'{palavra} {numero}'
    else:
        return s


dataframes = []
for root, dirs, files in os.walk(pasta):
    for arquivo in files:
        if arquivo.endswith(''):
            caminho_completo = os.path.join(root, arquivo)  
            print(f'Processando arquivo: {arquivo}') 
            # Leitura do arquivo
            try:
                df = pl.read_csv(caminho_completo, has_header=False).filter(pl.col('column_1').str.contains('TRIG'))
                df = df.with_columns(pl.col("column_1").str.split(" ").alias("split_column"))
                df = df.with_columns(pl.col("split_column").list.get(0).alias("TRIG"))
                df = df.with_columns(pl.col("split_column").list.get(1).cast(pl.Int64).alias("positrons"))
                df = df.with_columns(pl.col("split_column").list.get(2).cast(pl.Int64).alias("electrons"))
                df = df.with_columns(pl.col("split_column").list.get(3).cast(pl.Int64).alias("muons_plus"))
                df = df.with_columns(pl.col("split_column").list.get(4).cast(pl.Int64).alias("muons_minus"))
                df = df.with_columns(
                    (pl.col("positrons") + pl.col("electrons") + pl.col("muons_plus") + pl.col("muons_minus")).alias("total_particles")
                )

                # Adicionar uma coluna para identificar a simulação
                df = df.with_columns((pl.arange(0, df.height, eager=True) // 3).alias("simulation_id"))
                
                # Agregar por simulation_id e verificar se todos os detectores têm total_particles > 0
                valid_simulations = (
                    df.group_by("simulation_id")
                    .agg((pl.col("total_particles") > 0).all().alias("all_detectors_positive"))
                    .filter(pl.col("all_detectors_positive"))
                    .select("simulation_id")
                )
                
                # Filtrar o DataFrame original para manter apenas as simulações válidas
                df = df.filter(pl.col("simulation_id").is_in(valid_simulations["simulation_id"]))
                # Nome formatado do arquivo
                name = formatar_string(arquivo)
                
                # Transformação das contagens de partículas em listas e na média de contagens
                df = df.group_by('TRIG').agg(pl.col("total_particles")).sort(pl.col("TRIG").str.extract(r"TRIG([0-9]*)", 1).cast(int))
                df = df.with_columns(particles=pl.col('total_particles').list.mean())
                df = df.with_columns(
                    pl.lit(name.split()[0]).alias('composition'),
                    pl.lit(name.split()[1]).alias('energy')
                )
                df = df.drop('total_particles')
                
                # Adiciona o dataframe à lista
                dataframes.append(df)
            except Exception as e:
                print(f'Erro ao processar o arquivo {caminho_completo}: {e}') 


# Verifica se a lista de dataframes não está vazia
if dataframes:
    # Combina todos os dataframes em um único dataframe
    df_final = pl.concat(dataframes)
    # Embaralha o dataframe
    df_final = df_final.sample(fraction = 1.0, shuffle = True)
    # Exibe o dataframe final
    print(df_final)
else:
    print("Nenhum dataframe foi criado. Verifique se há arquivos válidos na pasta e se os arquivos estão no formato correto.")

Processando arquivo: carbon1E14
Processando arquivo: carbon1E15
Processando arquivo: carbon3.16E14
Processando arquivo: carbon3.16E15
Processando arquivo: iron1E14
Processando arquivo: iron1E15
Processando arquivo: iron3.16E14
Processando arquivo: iron3.16E15
Processando arquivo: nitrogen1E14
Processando arquivo: nitrogen1E15
Processando arquivo: nitrogen3.16E14
Processando arquivo: nitrogen3.16E15
Processando arquivo: oxygen1E14
Processando arquivo: oxygen1E15
Processando arquivo: oxygen3.16E14
Processando arquivo: oxygen3.16E15
Processando arquivo: photon1E14
Processando arquivo: photon1E15
Processando arquivo: photon3.16E14
Processando arquivo: photon3.16E15
Processando arquivo: proton1E14
Processando arquivo: proton1E15
Processando arquivo: proton3.16E14
Processando arquivo: proton3.16E15
shape: (72, 4)
┌───────┬────────────┬─────────────┬─────────┐
│ TRIG  ┆ particles  ┆ composition ┆ energy  │
│ ---   ┆ ---        ┆ ---         ┆ ---     │
│ str   ┆ f64        ┆ str         ┆ str

In [27]:
df_final.filter(
    (pl.col("TRIG") == "TRIG1") & (pl.col("composition") == 'Photon')
)

TRIG,particles,composition,energy
str,f64,str,str
"""TRIG1""",6.45082,"""Photon""","""1E14"""
"""TRIG1""",14.600601,"""Photon""","""3.16E14"""
"""TRIG1""",226.954128,"""Photon""","""3.16E15"""
"""TRIG1""",50.0,"""Photon""","""1E15"""


In [28]:
# Normalização da densidade
density_min = df_final.select(pl.col('particles').min()).to_numpy()[0, 0]
density_max = df_final.select(pl.col('particles').max()).to_numpy()[0, 0]
    
df_normalized = df_final.with_columns(
      ((pl.col('particles') - density_min) / (density_max - density_min)).alias('particles')
    )
df_normalized
#df_normalized.write_csv('data_neural_network.csv')

TRIG,particles,composition,energy
str,f64,str,str
"""TRIG3""",0.123445,"""Proton""","""1E15"""
"""TRIG3""",0.001475,"""Iron""","""1E14"""
"""TRIG2""",0.039818,"""Photon""","""3.16E14"""
"""TRIG2""",0.056805,"""Nitrogen""","""1E15"""
"""TRIG1""",0.306329,"""Oxygen""","""3.16E15"""
…,…,…,…
"""TRIG1""",0.075047,"""Oxygen""","""1E15"""
"""TRIG3""",0.670145,"""Photon""","""3.16E15"""
"""TRIG1""",0.020508,"""Oxygen""","""3.16E14"""


In [25]:
# Leitura do arquivo

arquivo = 'tratados_3det/photon/photon1E15'

df = pl.read_csv(arquivo, has_header= False).filter(pl.col('column_1').str.contains('TRIG'))

df = df.with_columns(pl.col("column_1").str.split(" ").alias("split_column"))
df = df.with_columns(pl.col("split_column").list.get(0).alias("TRIG"))
df = df.with_columns(pl.col("split_column").list.get(1).cast(pl.Int64).alias("positrons"))
df = df.with_columns(pl.col("split_column").list.get(2).cast(pl.Int64).alias("electrons"))
df = df.with_columns(pl.col("split_column").list.get(3).cast(pl.Int64).alias("muons_plus"))
df = df.with_columns(pl.col("split_column").list.get(4).cast(pl.Int64).alias("muons_minus"))
df = df.with_columns(
    (pl.col("positrons") + pl.col("electrons") + pl.col("muons_plus") + pl.col("muons_minus")).alias("total_particles")
)

# Adicionar uma coluna para identificar a simulação
df = df.with_columns((pl.arange(0, df.height, eager=True) // 3).alias("simulation_id"))

# Agregar por simulation_id e verificar se todos os detectores têm total_particles > 0
valid_simulations = (
    df.group_by("simulation_id")
    .agg((pl.col("total_particles") > 0).all().alias("all_detectors_positive"))
    .filter(pl.col("all_detectors_positive"))
    .select("simulation_id")
)

# Filtrar o DataFrame original para manter apenas as simulações válidas
filtered_df = df.filter(pl.col("simulation_id").is_in(valid_simulations["simulation_id"]))

print(filtered_df)

shape: (1_650, 9)
┌────────────┬────────────┬───────┬───────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ column_1   ┆ split_colu ┆ TRIG  ┆ positrons ┆ … ┆ muons_plus ┆ muons_min ┆ total_par ┆ simulatio │
│ ---        ┆ mn         ┆ ---   ┆ ---       ┆   ┆ ---        ┆ us        ┆ ticles    ┆ n_id      │
│ str        ┆ ---        ┆ str   ┆ i64       ┆   ┆ i64        ┆ ---       ┆ ---       ┆ ---       │
│            ┆ list[str]  ┆       ┆           ┆   ┆            ┆ i64       ┆ i64       ┆ i64       │
╞════════════╪════════════╪═══════╪═══════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ TRIG1 15   ┆ ["TRIG1",  ┆ TRIG1 ┆ 15        ┆ … ┆ 0          ┆ 0         ┆ 30        ┆ 5         │
│ 15 0 0     ┆ "15", …    ┆       ┆           ┆   ┆            ┆           ┆           ┆           │
│            ┆ "0"]       ┆       ┆           ┆   ┆            ┆           ┆           ┆           │
│ TRIG2 5 13 ┆ ["TRIG2",  ┆ TRIG2 ┆ 5         ┆ … ┆ 0          ┆ 0       